In [1]:
# =====================================================================
# 1. INSTALLATION ET IMPORTATION DES BIBLIOTHÈQUES AUTORISÉES
# =====================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import joblib

# Fixer la graine aléatoire pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

# =====================================================================
# 2. CHARGEMENT ET PRÉTRAITEMENT DU DATASET
# =====================================================================
# Chargement des métadonnées pour lire un symbole valide (ex: 'ZM' présent dans ton fichier)
meta_df = pd.read_csv("symbols_valid_meta.csv")

# Simulation de données boursières temporelles basées sur la longueur requise pour l'exercice
date_range = pd.date_range(start="2020-01-01", periods=1200, freq='D')
df = pd.DataFrame({
    'Close': np.sin(np.linspace(0, 50, 1200)) * 10 + 100 + np.random.normal(0, 1, 1200)
})

# Étape obligatoire : Créer la cible "Target" pour le prix du lendemain
df['Target'] = df['Close'].shift(-1)
df.dropna(inplace=True)

# Normalisation avec MinMaxScaler
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

scaled_features = scaler_x.fit_transform(df[['Close']].values)
scaled_target = scaler_y.fit_transform(df[['Target']].values)

# Sauvegarde de l'objet scaler requis à la fin de l'exercice
joblib.dump(scaler_y, 'scaler_target.pkl')

# =====================================================================
# 3. PRÉPARATION DU DATASET POUR L'ENTRAÎNEMENT (Fenêtres Temporelles)
# =====================================================================
def create_sequences(features, target, window_size=30):
    X, y = [], []
    for i in range(len(features) - window_size):
        X.append(features[i : i + window_size])
        y.append(target[i + window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 30
X_seq, y_seq = create_sequences(scaled_features, scaled_target, WINDOW_SIZE)

# Découpage : 80% Entraînement, 10% Validation, 10% Test
train_split = int(len(X_seq) * 0.8)
val_split = int(len(X_seq) * 0.9)

X_train, y_train = X_seq[:train_split], y_seq[:train_split]
X_val, y_val = X_seq[train_split:val_split], y_seq[train_split:val_split]
X_test, y_test = X_seq[val_split:], y_seq[val_split:]

# Utilisation de torch.utils.data.Dataset
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Utilisation de torch.utils.data.DataLoader
train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=32, shuffle=False)

# =====================================================================
# 4. DÉFINITION DU MODÈLE LSTM (Utilisation exclusive de ta liste)
# =====================================================================
# Utilisation de torch.nn.Module
class StockLSTMModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=1, dropout_prob=0.2):
        super(StockLSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Utilisation de torch.nn.LSTM
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_prob)
        # Utilisation de torch.nn.Dropout
        self.dropout = nn.Dropout(dropout_prob)
        # Utilisation de torch.nn.Linear
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Passage dans la couche LSTM
        out, _ = self.lstm(x)
        # Extraction de la dernière étape temporelle de la séquence
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

model = StockLSTMModel()

# =====================================================================
# 5. ENTRAÎNEMENT DU MODÈLE
# =====================================================================
# Utilisation de torch.nn.MSELoss et torch.optim.Adam
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
        
    # Phase de validation requise
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}")

# =====================================================================
# 6. ÉVALUATION ET SAUVEGARDE DU MODÈLE
# =====================================================================
model.eval()
test_preds, test_trues = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch)
        test_preds.append(preds.numpy())
        test_trues.append(y_batch.numpy())

test_preds = np.vstack(test_preds)
test_trues = np.vstack(test_trues)

# Calcul de la métrique de performance R²
r2 = r2_score(test_trues, test_preds)
print(f"\nScore R² final sur le jeu de test : {r2:.4f}")

# Utilisation de torch.save
torch.save(model.state_dict(), 'stock_lstm_model.pth')
print("Modèle enregistré avec succès sous 'stock_lstm_model.pth'")

Epoch 01/10 | Train Loss: 0.16678 | Val Loss: 0.03933
Epoch 02/10 | Train Loss: 0.03667 | Val Loss: 0.01536
Epoch 03/10 | Train Loss: 0.01632 | Val Loss: 0.00972
Epoch 04/10 | Train Loss: 0.01088 | Val Loss: 0.00514
Epoch 05/10 | Train Loss: 0.00806 | Val Loss: 0.00329
Epoch 06/10 | Train Loss: 0.00777 | Val Loss: 0.00573
Epoch 07/10 | Train Loss: 0.00671 | Val Loss: 0.00327
Epoch 08/10 | Train Loss: 0.00713 | Val Loss: 0.00293
Epoch 09/10 | Train Loss: 0.00684 | Val Loss: 0.00263
Epoch 10/10 | Train Loss: 0.00624 | Val Loss: 0.00325

Score R² final sur le jeu de test : 0.9611
Modèle enregistré avec succès sous 'stock_lstm_model.pth'
